In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd

#data classes
import xarray as xr
import pickle

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RadarComparison"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
spinup_hours = "0"

RunType = ("TRACER","WET","NSSL",spinup_hours)
# RunType = ("TRACER","DRY","NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = ("TRACER","WET","TEMPO",spinup_hours)
# RunType = ("TRACER","DRY","TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#Importing ERA5 Data Loading Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","ERA5_Data"))
from CLASSES_ERA5DataLoading import ERA5DataLoading_Class,ERA5DataLoading_Class_gdex

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
########################
#DATA INFORMATION

In [ ]:
#DATA CITATION
# Atmospheric Radiation Measurement (ARM) user facility. 2021. X-Band Scanning ARM Cloud Radar (XSACRCFRQC), 2022-06-09 to 2022-07-02, ARM Mobile Facility (HOU) Houston, TX; AMF1 (main site for TRACER) (M1). Compiled by Y. Feng, A. Matthews, E. Schuman, K. Johnson, I. Lindenmaier, V. Castro and T. Wendler. ARM Data Center. Data set accessed 2025-11-05 at http://dx.doi.org/10.5439/2001296.

#Globus Download Link
# https://urldefense.com/v3/__https://app.globus.org/file-manager?origin_id=ba87aabe-30f6-433d-b4a5-19434c595e0f&origin_path=*rosemana1*261781*__;Ly8v!!PvDODwlR4mBZyAb0!REKAOzHJNvJk50CY5Pjl135CV83BhArwtdyMDuBM-28KqreBug8Xb5Mc3MLgy_p9PUiWe2uVXq-EfUtLcGdH5w$

In [ ]:
#DATA CITATION
# Atmospheric Radiation Measurement (ARM) user facility. 2021. Ka-Band Scanning ARM Cloud Radar (KASACRCFRQC), 2022-06-08 to 2022-07-02, ARM Mobile Facility (HOU) Houston, TX; AMF1 (main site for TRACER) (M1). Compiled by I. Lindenmaier, K. Johnson, D. Nelson, A. Matthews, T. Wendler, V. Melo de Castro, M. Rocque and Y. Feng. ARM Data Center. Data set accessed 2025-11-05 at http://dx.doi.org/10.5439/1877338.

# https://armgov.svcs.arm.gov/capabilities/instruments/kasacr

#Globus Download Link
#https://urldefense.com/v3/__https://app.globus.org/file-manager?origin_id=ba87aabe-30f6-433d-b4a5-19434c595e0f&origin_path=*rosemana1*261893*__;Ly8v!!PvDODwlR4mBZyAb0!UJrKWg_xaYuaaa_iaY91TCVMB_neSskXsDKHtPPCG7Ix6sEmiEvnngTUrYuV18LudZqxB3eHyTDuCHSZ3o3zrg$

In [ ]:
#LOADING RADAR CLASS
if spinup_hours == "0" and ModelData_NSSL.region == "TRACER":
    dateString = '2022-06-30_2022-07-03'
else:
    dateString = f"{ModelData_NSSL.simulationDates[0]}_{ModelData_NSSL.simulationDates[-1]}"

RadarData_MRMS = RadarData_MRMS_Class(ModelData_NSSL,
                                      fileDirectory=os.path.join(DirectoryManager.dataDirectory,
                                                                 "Observation_Data/TRACER/MRMS_RadarData",
                                                                 dateString))

In [ ]:
##########################
#DATA LOADING FUNCTIONS

In [ ]:
#Converting timeStrings
def ConvertTimeStringtoDateTime(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    """
    return datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
def ConvertTimeStringtoTimeTitle(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    Formatted for use as a plot title.
    """
    # Parse the custom format to a datetime object
    dt = datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
    # Format it to 'YYYY-MM-DD HH:MM:SS'
    return dt.strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
RadarDataMask = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_NSSL)

In [ ]:
def GetData(t):
    timeString = ModelData_NSSL.timeStrings[t]
    timeString_datetime = ConvertTimeStringtoDateTime(timeString)
    
    #Loading Model Radar
    modelRadarData_NSSL = ModelData_NSSL.GetDataTimestep_diag(t)["refl10cm_1km"]
    modelRadarData_TEMPO = ModelData_TEMPO.GetDataTimestep_diag(t)["refl10cm_1km"]
    modelRadarTimeTitle = ConvertTimeStringtoTimeTitle(timeString)
    
    #Loading Observational Radar
    radarData, nearestFilePath = RadarData_MRMS.LoadClosestMRMSFile(target_time=timeString_datetime)
    radarTimeTitle = pd.to_datetime(radarData['time'].data[0]).strftime("%Y-%m-%d %H:%M:%S")
    radarData=radarData.isel(time=0)
    radarData_interp = InterpolateRadarData(radarData=radarData, modelData=modelRadarData_NSSL)

    #Getting Model MSLP Data
    mslpData_NSSL = ModelData_NSSL.GetDataTimestep_diag(t)['mslp']/1e2
    mslpData_TEMPO = ModelData_TEMPO.GetDataTimestep_diag(t)['mslp']/1e2

    #Getting ERA5 MSLP Data
    mslp_ERA5_alltimes = ERA5DataLoading_Class_gdex.LoadERA5Data(timeString, ModelData_NSSL, DirectoryManager)
    mslp_ERA5 = ERA5DataLoading_Class_gdex.SelectNearestERA5Time(mslp_ERA5_alltimes, timeString)/1e2
    # mslp_ERA5_alltimes = ERA5DataLoading_Class.LoadERA5Data(DirectoryManager, ModelData_NSSL, variableName='msl',dataType='Surface')
    # mslp_ERA5 = ERA5DataLoading_Class.SelectNearestERA5Time(mslp_ERA5_alltimes, ModelData_NSSL.timeStrings[t])/1e2  

    # Applying RadarDataMask
    modelRadarData_NSSL = modelRadarData_NSSL.where(RadarDataMask == True)
    modelRadarData_TEMPO = modelRadarData_TEMPO.where(RadarDataMask == True)
    radarData_interp = radarData_interp.where(RadarDataMask == True)
    
    return (modelRadarData_NSSL,modelRadarData_TEMPO,modelRadarTimeTitle, 
            radarData_interp,radarTimeTitle, 
            timeString,
            mslpData_NSSL,mslpData_TEMPO,mslp_ERA5)

In [ ]:
def FixLatLon_RadarData(radarData):        
    radarData = radarData.isel(latitude=slice(None, None, -1))

    radarData = radarData.assign_coords(
        longitude=((radarData.longitude + 180) % 360) - 180
    )
    return radarData

def ReturnLatLon_RadarData(radarData):
    # Fix latitude order
    radarData_fixed = radarData.isel(latitude=slice(None, None, -1))

    # Fix longitude convention
    radarData_fixed = radarData_fixed.assign_coords(
        longitude=radarData.longitude+360
    )

    return radarData_fixed


def InterpolateRadarData(radarData,modelData):
    radarData = FixLatLon_RadarData(radarData)
    
    radarData_interp = radarData.interp(
        latitude=modelData.latitude,
        longitude=modelData.longitude,
        method="linear"
    )
    return radarData_interp

In [ ]:
##########################
#PLOTTING FUNCTIONS

In [ ]:
def MakePlot(modelRadarData_NSSL,modelRadarData_TEMPO,modelRadarTimeTitle, 
             radarData,radarTimeTitle,
             mslpData_NSSL,mslpData_TEMPO,mslp_ERA5):
    
    fig, axes = RadarPlotting_Class.CreateMapAxes(nrows=1,ncols=3,
                                                  figsize=(16,8))
    
    #Plotting ModelRadar
    #nssl
    axis = axes[0,0]
    lat = modelRadarData_NSSL['latitude']
    lon = modelRadarData_NSSL['longitude']
    contourPlot = RadarPlotting_Class.PlotReflectivity(axis, lat,lon,modelRadarData_NSSL,dataName="NSSL",timeTitle=modelRadarTimeTitle)

    #Adding MSLP Contours
    cs1 = axis.contour(lon, lat, mslpData_NSSL, 
                       colors='black', levels=10, linewidths=1.0, alpha=0.35, zorder=10)
    labels = axis.clabel(cs1, inline=True, fontsize=8, fmt="%.0f",
                colors='black',zorder=11)
    for label in labels:
        label.set_alpha(1)
    
    #tempo
    axis = axes[0,2]
    lat = modelRadarData_TEMPO['latitude']
    lon = modelRadarData_TEMPO['longitude']
    RadarPlotting_Class.PlotReflectivity(axis, lat,lon,modelRadarData_TEMPO,dataName="TEMPO",timeTitle=modelRadarTimeTitle)

    #Adding MSLP Contours
    cs1 = axis.contour(lon, lat, mslpData_TEMPO, 
                       colors='black', levels=10, alpha=0.35, linewidths=1.0,zorder=10)
    labels = axis.clabel(cs1, inline=True, fontsize=8, fmt="%.0f",
                colors='black',zorder=11)
    for label in labels:
        label.set_alpha(1)
    
    #Plotting Observational Radar
    #mrms data
    axis = axes[0,1]
    lat = radarData['latitude'].data
    lon = radarData['longitude'].data-360
    
    RadarPlotting_Class.PlotReflectivity(axis, lat,lon,radarData,dataName="MRMS",timeTitle=radarTimeTitle)

    #Adding MSLP Contours
    cs1 = axis.contour(mslp_ERA5.longitude, mslp_ERA5.latitude, mslp_ERA5, 
                       colors='black', levels=10, alpha=0.35, linewidths=1.0,zorder=10)
    labels = axis.clabel(cs1, inline=True, fontsize=8, fmt="%.0f",
                colors='black',zorder=11)
    for label in labels:
        label.set_alpha(1)
    
    #Adding Colorbar
    colorBar = RadarPlotting_Class.AddSharedColorbar(fig, contourPlot)


    return fig

In [ ]:
##########################
#PLOTTING

In [ ]:
t=92
(modelRadarData_NSSL,modelRadarData_TEMPO,modelRadarTimeTitle, 
            radarData_interp,radarTimeTitle, 
            timeString,
            mslpData_NSSL,mslpData_TEMPO,mslp_ERA5) = GetData(t)
radarData_interp = ReturnLatLon_RadarData(radarData_interp) #not necessary unless plotting below

In [ ]:
fig = MakePlot(modelRadarData_NSSL,modelRadarData_TEMPO,modelRadarTimeTitle, 
               radarData_interp,radarTimeTitle,
               mslpData_NSSL,mslpData_TEMPO,mslp_ERA5)

In [ ]:
##########################
#CALCULATING FUNCTIONS

In [ ]:
# Pulkkinen, S., D. Nerini, A. Perez Hortal, C. Velasco-Forero, U. Germann, A. Seed, and L. Foresti, 2019: 
# Pysteps: an open-source Python library for probabilistic precipitation nowcasting (v1.0). Geosci. Model Dev., 12 (10), 4185–4219, doi:10.5194/gmd-12-4185-2019.
# Imhoff, R.O., L. De Cruz, W. Dewettinck, C.C. Brauer, R. Uijlenhoet, K-J. van Heeringen, C. Velasco-Forero, D. Nerini, M. Van Ginderachter, and A.H. Weerts, 2023: 
# Scale-dependent blending of ensemble rainfall nowcasts and NWP in the open-source pysteps library. Q J R Meteorol Soc., 1-30, doi: doi:10.1002/qj.4461.
# https://pysteps.readthedocs.io/en/stable/generated/pysteps.verification.spatialscores.fss.html

# pip install pysteps
from pysteps.verification.spatialscores import fss

def CalculateFSS_pysteps(forecast,observation,threshold,scale):
    fs_score = fss(X_f=forecast, X_o=observation, thr=threshold, scale=scale)
    return fs_score*100

In [ ]:
# Leeuwenburg, T., Loveday, N., Ebert, E. E., Cook, H., Khanarmuei, M., Taggart, R. J., Ramanathan, N., Carroll, M., Chong, S., Griffiths, A., & Sharples, J. (2024). 
# scores: A Python package for verifying and evaluating models and predictions with xarray. Journal of Open Source Software, 9(99), 6889. https://doi.org/10.21105/joss.06889
# https://scores.readthedocs.io/en/2.0.0/tutorials/Fractions_Skill_Score.html

# pip install scores
from scores.spatial import fss_2d_single_field
from scores.fast.fss.typing import FssComputeMethod

def CalculateFSS_scores(forecast,observation,threshold,window_size):
    compute_method = FssComputeMethod.NUMPY
    threshold_operator = np.greater_equal
    # threshold_operator = np.greater
    
    fs_score = fss_2d_single_field(
        forecast,
        observation,
        event_threshold=threshold,
        window_size=window_size,           # same interpretation as 'scale'
        threshold_operator=threshold_operator,
        compute_method=compute_method # default and fastest
    )
    return fs_score*100

In [ ]:
scale=3
def Run_FSS(forecast,observation, thresholds=[0,20,40,65], printstatement=False):
    #scale: nxn pixels
    fs_scores = []
    for threshold in thresholds:
        # fs_score = CalculateFSS_pysteps(forecast=forecast,observation=observation,threshold=threshold,scale=scale)
        # if printstatement==True:
            # print(f"FSS = {fs_score:.2f}% for threshold = {threshold} dBZ")
        
        fs_score = CalculateFSS_scores(forecast=forecast,observation=observation,threshold=threshold,window_size=(scale,scale))
        if printstatement==True:
            print(f"FSS = {fs_score:.2f}% for threshold = {threshold} dBZ")
        fs_scores.append(fs_score)
    return fs_scores, thresholds

In [ ]:
##########################
#CALCULATING FOR SINGLE TIMESTEP

In [ ]:
# #COMPARING MRMS AND NSSL FIELDS
# forecast = modelRadarData_NSSL.data
# observation = radarData_interp.data

# print("COMPARING MRMS AND NSSL FIELDS","\n")
# fs_scores, thresholds = Run_FSS(forecast,observation, printstatement=True)

In [ ]:
# #COMPARING MRMS AND TEMPO FIELDS
# forecast = modelRadarData_TEMPO.data
# observation = radarData_interp.data

# print("COMPARING MRMS AND TEMPO FIELDS","\n")
# fs_scores, thresholds = Run_FSS(forecast,observation, printstatement=True)

In [ ]:
# #COMPARING NSSL AND TEMPO FIELDS
# forecast = modelRadarData_NSSL.data
# observation = modelRadarData_TEMPO.data

# print("COMPARING NSSL AND TEMPO FIELDS","\n")
# fs_scores, thresholds = Run_FSS(forecast,observation, printstatement=True)

In [ ]:
##########################
#LOADING DATA

In [ ]:
##########################
#CALCULATING FOR ALL TIMESTEPS

In [ ]:
def LoadFractionSkillScore(ModelData):
    """
    Build the FSS filename using ModelData and load the .pkl file.
    Creates output directory if needed.
    Returns (fullFilePath, loadedData or None).
    """

    # Build file name
    fileName = (
        f"FractionSkillScore_{ModelData.region}_"
        f"{ModelData.case}_spinup{ModelData.spinup_hours}hrs.pkl"
    )

    # Build directory for FSS output
    outputDir = os.path.join(
        DirectoryManager.GetOutputDirectory(codeType, dataType),
        "FractionSkillScore"
    )
    os.makedirs(outputDir, exist_ok=True)

    # Full path to the .pkl file
    fullFilePath = os.path.join(outputDir, fileName)

    # Try to load existing file
    if os.path.exists(fullFilePath):
        with open(fullFilePath, "rb") as f:
            return fullFilePath, pickle.load(f)

    # No cached file found
    return fullFilePath, None


In [ ]:
def RunCode():

    # -------------------------------------------------------
    # 1. Load existing FSS file (or get path for saving)
    # -------------------------------------------------------
    fullFilePath, loadedData = LoadFractionSkillScore(ModelData_NSSL)

    if loadedData is not None:
        print(f"Loaded precomputed FractionSkillScore scores from {fullFilePath}")

        scores_array_NSSL  = loadedData["scores_array_NSSL"]
        scores_array_TEMPO = loadedData["scores_array_TEMPO"]
        thresholds         = loadedData["thresholds"]

        return scores_array_NSSL, scores_array_TEMPO, thresholds

    # -------------------------------------------------------
    # 2. Compute FSS because no cached file exists
    # -------------------------------------------------------
    print("No cached FSS file found — computing FSS scores...")

    fs_scores_NSSL = []
    fs_scores_TEMPO = []

    for t in tqdm(range(ModelData_NSSL.Ntime)):

        # Getting Data
        (modelRadarData_NSSL, modelRadarData_TEMPO, modelRadarTimeTitle,
         radarData_interp, radarTimeTitle,
         timeString,
         mslpData_NSSL, mslpData_TEMPO, mslp_ERA5) = GetData(t)
        
        # Calculating
        forecast_1 = modelRadarData_NSSL.data
        forecast_2 = modelRadarData_TEMPO.data
        observation = radarData_interp.data

        fs_scores1, thresholds = Run_FSS(forecast_1, observation, printstatement=False)
        fs_scores2, _ = Run_FSS(forecast_2, observation, printstatement=False)
        fs_scores_NSSL.append(fs_scores1)
        fs_scores_TEMPO.append(fs_scores2)

    scores_array_NSSL = np.array(fs_scores_NSSL)
    scores_array_TEMPO = np.array(fs_scores_TEMPO)

    # -------------------------------------------------------
    # 3. Save newly computed FSS results
    # -------------------------------------------------------
    data_to_save = {
        "scores_array_NSSL": scores_array_NSSL,
        "scores_array_TEMPO": scores_array_TEMPO,
        "thresholds": thresholds,
    }

    with open(fullFilePath, "wb") as f:
        pickle.dump(data_to_save, f)

    print(f"Saved FSS results to {fullFilePath}")

    return scores_array_NSSL, scores_array_TEMPO, thresholds


In [ ]:
[scores_array_NSSL, scores_array_TEMPO, thresholds] = RunCode()

In [ ]:
##########################
#PLOTTING FUNCTIONS

In [ ]:
#HELPER FUNCTIONS
def SetXLimitsDatetime(ax, time_array):
    """
    Ensures datetime x-axis starts and ends exactly at the first and last time values.
    Works for both datetime.datetime and np.datetime64 arrays.
    """
    # Convert to Matplotlib’s internal float format if needed
    times = np.asarray(time_array)
    if np.issubdtype(times.dtype, np.datetime64):
        times = date2num(times)
    elif isinstance(times[0], (object,)):
        try:
            times = date2num(times)
        except Exception:
            pass

    ax.set_xlim(times.min(), times.max())

In [ ]:
def MakeFSSPlot(scores_array_NSSL, scores_array_TEMPO, thresholds):
    """
    Plot FSS time series for NSSL (solid) and TEMPO (dashed)
    with two legends: thresholds and model line styles.
    """

    #time axis
    time_strings = ModelData_NSSL.timeStrings
    times = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in time_strings]

    # Colors for thresholds
    colors = [
        "black",   # threshold 0
        "blue",    # threshold 1
        "purple",  # threshold 2
        "red",     # threshold 3
    ]

    # Create figure & axis
    fig, ax = plt.subplots(figsize=(10, 6))

    # ------------------------------------------------------
    # Plotting lines
    # ------------------------------------------------------
    for i in range(scores_array_NSSL.shape[1]):
        ax.plot(
            times,
            scores_array_NSSL[:, i],
            linestyle='solid',
            color=colors[i],
            label=f">= {thresholds[i]} dBZ"
        )

        ax.plot(
            times,
            scores_array_TEMPO[:, i],
            linestyle='dashed',
            color=colors[i]
        )

    # Axis limits (flush last tick to edge)
    ax.set_xlim(0, ModelData_NSSL.Ntime - 1)
    ax.set_ylim(bottom=0)

    # ------------------------------------------------------
    # Legend 1: Threshold colors (left)
    # ------------------------------------------------------
    threshold_legend = [
        Line2D(
            [0], [0],
            color=colors[i],
            linestyle='solid',
            linewidth=2,
            label=f">= {thresholds[i]} dBZ"
        )
        for i in range(len(thresholds))
    ]

    leg1 = ax.legend(
        handles=threshold_legend,
        title="Thresholds",
        loc="upper left"
    )

    # ------------------------------------------------------
    # Legend 2: Model styles (top center)
    # ------------------------------------------------------
    model_legend = [
        Line2D([0], [0], color="black", linestyle='solid', linewidth=2, label="NSSL"),
        Line2D([0], [0], color="black", linestyle='dashed', linewidth=2, label="TEMPO"),
    ]

    leg2 = ax.legend(
        handles=model_legend,
        title="Models",
        loc="upper center",
        bbox_to_anchor=(0.5, 1.15),
        ncol=2,
        frameon=False
    )

    # Add first legend back so both show
    ax.add_artist(leg1)

    # ------------------------------------------------------
    # Labels, grid, title
    # ------------------------------------------------------
    # ax.set_xlabel("Time") 
    ax.set_ylabel("FSS (%)")
    ax.set_title(f"Plot of Fraction Skill Score: window = ({scale},{scale})")
    ax.grid(True)
    SetXLimitsDatetime(ax, time_array=times)

    return fig

In [ ]:
def GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory):
    outputSubDirectory = f"{ModelData_1.region}_{ModelData_1.case}_{ModelData_1.spinup_hours}hrs"
    
    outputFilePath = os.path.join(
        outputPlottingDirectory,
        "FractionSkillScore",
        outputSubDirectory)
    os.makedirs(outputFilePath, exist_ok=True)
    return outputFilePath

def SaveFigure(fig, ModelData_1,ModelData_2):
    """
    Saves a figure to corresponding directory.
    """
    # --- Define output subdirectory and file path ---
    outputFilePath = GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory)
    outputFile = os.path.join(outputFilePath,f"FractionSkillScore_{ModelData_1.mpType}vsMRMSvs{ModelData_2.mpType}.png")

    # --- Save figure ---
    fig.savefig(outputFile, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

In [ ]:
##########################
#PLOTTING

In [ ]:
fig = MakeFSSPlot(scores_array_NSSL, scores_array_TEMPO, thresholds)
SaveFigure(fig, ModelData_NSSL, ModelData_TEMPO)